### Exercice 3 : Two Continuous Stirred Tank Reactors (CSTRs) in Series

We consider a system of two continuous stirred tank reactors (CSTRs) operating at steady state.

Given data:
Flow rate: $F = 100 \, \text{L/h}$
Inlet substrate concentration: $S_0 = 10 \, \text{g/L}$
Yield coefficient: $Y_{X/S} = 0.5 \, \text{g cells/g substrate}$
Maximum growth rate: $\mu_{\max} = 1 \, \text{h}^{-1}$
Monod constant: $K_S = 0.75 \, \text{g/L}$

Assumptions:
Steady-state operation
No maintenance energy
Monod kinetics:

$$
\mu(S) = \mu_{\max} \frac{S}{K_S + S}
$$

Model equations: We consider two CSTRs in series at steady state.

Dilution rate

$$
D_i = \frac{F}{V_i}
$$

Monod kinetics

$$
\mu(S) = \mu_{\max} \frac{S}{K_S + S}
$$

---

Reactor 1 balances

Biomass balance:

$$
0 = \mu(S_1) X_1 - D_1 X_1
$$

Substrate balance:

$$
0 = D_1 (S_0 - S_1) - \frac{1}{Y_{X/S}} \mu(S_1) X_1
$$

---

Reactor 2 balances

Biomass balance:

$$
0 = \mu(S_2) X_2 - D_2 X_2
$$

Substrate balance:

$$
0 = D_2 (S_1 - S_2) - \frac{1}{Y_{X/S}} \mu(S_2) X_2
$$

In [1]:
import numpy as np
from scipy.optimize import fsolve

In [2]:
# Parameters
F = 100  # L/h
S0 = 10  # g/L
Yxs = 0.5  # g/g
mu_max = 1  # h^-1
Ks = 0.75  # g/L

In [3]:
def system(vars, V1, V2):
    S1, X1, S2, X2 = vars
    
    D1 = F / V1
    D2 = F / V2
    
    mu1 = mu_max * S1 / (Ks + S1)
    mu2 = mu_max * S2 / (Ks + S2)
    
    
    eq1 = mu1 - D1
    eq3 = mu2 - D2
    
    
    eq2 = D1 * (S0 - S1) - (1 / Yxs) * mu1 * X1
    eq4 = D2 * (S1 - S2) - (1 / Yxs) * mu2 * X2
    
    return [eq1, eq2, eq3, eq4]

To avoid convergence to the trivial washout solution (X = 0), the condition μ = D is directly enforced for steady-state with biomass.

In [4]:
def solve_case(V1, V2):
    initial_guess = [0.1, 5.0, 0.1, 1.0]
    sol = fsolve(system, initial_guess, args=(V1, V2))
    return sol

A physically consistent initial guess is used to improve convergence of the nonlinear solver.  
At steady state with non-zero biomass, each reactor satisfies:

$$
\mu_i = D_i
$$

Using the Monod model:

$$
\mu = \mu_{\max} \frac{S}{K_s + S}
\;\;\Rightarrow\;\;
S_i = \frac{D_i K_s}{\mu_{\max} - D_i}
$$

Biomass is then estimated from substrate balances:

$$
X_1 = Y_{xs}(S_0 - S_1), \quad X_2 = Y_{xs}(S_1 - S_2)
$$

This provides a realistic starting point close to the expected steady state.

The initial guess helps numerical convergence but does not determine the physical validity of the solution.

Some configurations inherently lead to:

$$
S_2 > S_1 \;\Rightarrow\; X_2 < 0
$$

Such results are non-physical and arise from the model and operating conditions, not from the choice of initial guess.

Note: The nonlinear system admits multiple solutions, including the trivial washout solution (X = 0). 

In [5]:
cases = {
    "a": (800, 200),
    "b": (200, 800),
    "c": (900, 100),
    "d": (100, 900),
}

results = {}

for key, (V1, V2) in cases.items():
    S1, X1, S2, X2 = solve_case(V1, V2)
    results[key] = (S1, X1, S2, X2)

results

/var/folders/ps/2x_1r_2s4t5f7dl97vmk4psr0000gn/T/ipykernel_10133/2460296800.py:3: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last five Jacobian evaluations.
  sol = fsolve(system, initial_guess, args=(V1, V2))
/var/folders/ps/2x_1r_2s4t5f7dl97vmk4psr0000gn/T/ipykernel_10133/2460296800.py:3: RuntimeWarning: The number of calls to function has reached maxfev = 1000.
  sol = fsolve(system, initial_guess, args=(V1, V2))


{'a': (0.10714285714285718,
  4.946428571423448,
  0.7500000000000686,
  -0.321428571426374),
 'b': (0.7500000000000445,
  4.6250000000306875,
  0.10714285714285693,
  0.32142857132253505),
 'c': (0.0937499999999991,
  4.953125000000046,
  783411915.0598491,
  -391705957.8580496),
 'd': (591067.0882924355,
  -295528.9191398732,
  0.0937500044769136,
  295533.48472668265)}

In [6]:
for k, (S1, X1, S2, X2) in results.items():
    print(f"Case {k}:")
    print(f"  S1 = {S1:.3f}, X1 = {X1:.3f}")
    print(f"  S2 = {S2:.3f}, X2 = {X2:.3f}")
    print()

Case a:
  S1 = 0.107, X1 = 4.946
  S2 = 0.750, X2 = -0.321

Case b:
  S1 = 0.750, X1 = 4.625
  S2 = 0.107, X2 = 0.321

Case c:
  S1 = 0.094, X1 = 4.953
  S2 = 783411915.060, X2 = -391705957.858

Case d:
  S1 = 591067.088, X1 = -295528.919
  S2 = 0.094, X2 = 295533.485



Single reactor (1000 L)

For a single CSTR:

$$
0 = \mu(S) X - D X
$$

$$
0 = D (S_0 - S) - \frac{1}{Y_{X/S}} \mu(S) X
$$

In [7]:
def single_reactor(vars):
    S, X = vars
    D = F / 1000
    
    mu = mu_max * S / (Ks + S)
    
    eq1 = mu - D
    
    eq2 = D * (S0 - S) - (1 / Yxs) * mu * X
    
    return [eq1, eq2]

sol_single = fsolve(single_reactor, [1, 1])
S, X = sol_single

print(f"Single reactor:")
print(f"S = {S:.3f}, X = {X:.3f}")

Single reactor:
S = 0.083, X = 4.958


1.Single Reactor

The single continuous stirred-tank reactor (CSTR) was solved under steady-state conditions using Monod kinetics.
Results:
S = 0.083  
X = 4.958  

Interpretation:
This solution is physically consistent:
Substrate is low so efficient consumption
Biomass is positive so biologically feasible state
The system satisfies the steady-state condition:

$$
\mu = D
$$

This corresponds to a stable non-washout equilibrium.

---

2.Two-Stage Reactor System

The two reactors in series were analyzed for different volume configurations, affecting dilution rates:

$$
D_i = \frac{F}{V_i}
$$

---
Case (a): (V1 = 800, V2 = 200)

- S₁ = 0.107, X₁ = 4.946  
- S₂ = 0.750, X₂ = -0.321  

Analysis:
Reactor 1 is physically valid
Reactor 2 gives negative biomass so non-physical 
Indicates breakdown of biological steady-state assumption in R2

---
Case (b): (V1 = 200, V2 = 800)

S₁ = 0.750, X₁ = 4.625  
S₂ = 0.107, X₂ = 0.321  

Analysis:
Both reactors are physically consistent 
Efficient substrate transfer between reactors
This is the most stable and realistic configuration

---
Case (c): (V1 = 900, V2 = 100)
S₁ = 0.094, X₁ = 4.953  
S₂ = 783411915.060, X₂ = -391705957.858

Analysis:
Severe numerical divergence 
Non-physical values (huge substrate, negative biomass)
System is ill-conditioned due to extreme dilution imbalance
Solver (fsolve) diverges outside realistic biological regime

---
Case (d): (V1 = 100, V2 = 900)
S₁ = 591067.088, X₁ = -295528.919
S₂ = 0.094, X₂= 295533.485 

Analysis:
Reactor 1 is non-physical (washout: X < 0)
Reactor 2 remains stable
Strong dilution in R1 causes system instability

---

3.Comparison with Single Reactor

| System      | Substrate S        | Biomass X         | Physical validity |
|------------|--------------------|-------------------|------------------|
| Single     | 0.083              | 4.958             |  stable |
| Case (a)   | mixed              | mixed             | partially invalid |
| Case (b)   | 0.750 / 0.107      | 4.625 / 0.321     | stable |
| Case (c)   | diverging          | diverging         |  unstable |
| Case (d)   | diverging          | diverging         | unstable |

---

4.Physical Interpretation

The system is highly sensitive to dilution rates:

$$
D = \frac{F}{V}
$$

Only balanced configurations (especially Case b) produce physically meaningful steady states.
Non-physical results arise from:
Negative biomass values ($X < 0$)
Diverging substrate concentrations
Ill-conditioned nonlinear system
Limitations of Newton-based solver (`fsolve`)
Operating regimes outside Monod validity

---

5.Conclusion

The single reactor provides a stable and physically consistent benchmark
The two-stage system is highly sensitive to reactor volume distribution
Only specific configurations (Case b) remain fully realistic
Extreme imbalances lead to washout or numerical divergence

 Overall, proper scaling of dilution rates is essential for stable bioreactor design.